Code pour faire fonctionner pyspark (ne pas exécuter si pas besoin)

In [1]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [2]:
import sys
# Affiche le chemin de l'exécutable Python utilisé par ce notebook
print(sys.executable)

c:\Users\etien\miniconda3\envs\nf26\python.exe


In [3]:
import os
from pyspark.sql import SparkSession

# Arrêter la session Spark existante pour appliquer les nouveaux paramètres
if 'spark' in locals():
    spark.stop()

# Définir explicitement le chemin de l'exécutable Python pour Spark
# Ce chemin correspond à l'environnement du kernel du notebook
python_path = r'c:\Users\etien\miniconda3\envs\nf26\python.exe'
os.environ['PYSPARK_PYTHON'] = python_path
os.environ['PYSPARK_DRIVER_PYTHON'] = python_path

# Recréer la SparkSession avec la configuration mémoire et la configuration du worker
spark = (
    SparkSession.builder
    .appName("GHG-Inventory-ETL")
    .config("spark.driver.memory", "8g")
    .config("spark.python.worker.exec", python_path)
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print(f"PYSPARK_PYTHON est maintenant défini sur : {os.environ.get('PYSPARK_PYTHON')}")
print("Nouvelle session Spark créée avec succès.")

PYSPARK_PYTHON est maintenant défini sur : c:\Users\etien\miniconda3\envs\nf26\python.exe
Nouvelle session Spark créée avec succès.


In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *
from datetime import date, timedelta
from geopy.geocoders import Nominatim
from geopy.distance import geodesic

# Sert à calculer les distances entre les villes
geolocator = Nominatim(user_agent="BGES_app")

## Partie ETL

#### Fonctions d'homogénéisation de la langue des fonctions et missions

In [5]:
def clean_langue_fonction(df, site):
    match site:
        case "BERLIN":
            map_fonctions = {
                "Ökonom": "Economist",
                "Führungskraft": "Business Executive",
                "Personalleiter": "HRD",
                "Computeringenieur": "Computer Engineer",
                "Dateningenieur": "Data Engineer",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)
        case "PARIS":
            map_fonctions = {
                "Ingénieur Informaticien": "Computer Engineer",
                "Ingénieur Data": "Data Engineer",
                "Economiste": "Economist",
                "DRH": "HRD",
                "Cadre": "Business Executive",
            }
            df["FONCTION_PERSONNEL"] = df["FONCTION_PERSONNEL"].replace(map_fonctions)


In [6]:
def clean_langue_mission(df, site):
    match site:
        case "BERLIN":
            map_type_mission = {
                "Geschäftstreffen": "Business Meeting",
                "Konferenz": "Conference",
                "Schulung": "Vocational Training",
                "Meeting": "Team Meeting",
                "Entwicklung": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)
        case "PARIS":
            map_type_mission = {
                "Conférence": "Conference",
                "Formation": "Vocational Training",
                "Réunion": "Team Meeting",
                "Rencontre entreprises": "Business Meeting",
                "Développement": "Development",
            }
            df["TYPE_MISSION"] = df["TYPE_MISSION"].replace(map_type_mission)

#### Fonction d'homogénéisation des fuseaux horaires

In [8]:
def clean_date(df, site, date_col):
    """
    Convertit la colonne de dates d'un site donné vers UTC+2 (heure de Paris).
    Les dates sont supposées être en heure locale du site.
    """
    # Mapping site -> fuseau horaire IANA
    site_tz = {
        "PARIS":      "Europe/Paris",
        "BERLIN":     "Europe/Berlin",
        "LONDON":     "Europe/London",
        "NEWYORK":    "America/New_York",
        "LOSANGELES": "America/Los_Angeles",
        "SHANGHAI":   "Asia/Shanghai",
    }

    tz = site_tz[site]

    # S'assurer que la colonne est bien en datetime
    df[date_col] = pd.to_datetime(df[date_col])

    # Localiser la date dans le fuseau du site, puis convertir vers Paris (UTC+2 en été)
    df[date_col] = (
        df[date_col]
          .dt.tz_localize(tz, ambiguous=False, nonexistent='shift_forward') # on dit "cette heure est en heure locale du site"
          # le param ambiguous est utilisé pour les heures du changement d'heure
          .dt.tz_convert("Europe/Paris")  # on la convertit vers Paris
          .dt.tz_localize(None)           # on retire l'info de fuseau pour rester naive
    )

#### Chargement de l'ensemble des données du personnel

In [9]:
def extract_personnel():
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")

    personnel_dfs = []

    # On charge la liste du personnel de chaque site dans un dataframe
    for site in sites:
        file_path = base_path / f"BDD_BGES_{site}" / f"PERSONNEL_{site}.txt"
        df = pd.read_csv(str(file_path), sep=';')
        clean_langue_fonction(df, site) 
        df['ID_SITE'] = site
        
        personnel_dfs.append(df)

    # On combine les dataframes 
    personnel_df = pd.concat(personnel_dfs)
  
    # On sélectionne uniquement les colonnes nécessaires
    personnel_final_df = personnel_df[['ID_PERSONNEL','FONCTION_PERSONNEL', 'ID_SITE']]
    return personnel_final_df

In [10]:
personnel_df = extract_personnel()
personnel_df.head()

,ID_PERSONNEL,FONCTION_PERSONNEL,ID_SITE
0,KeyPers_Paris_1230000,Business Executive,PARIS
1,KeyPers_Paris_1230001,HRD,PARIS
2,KeyPers_Paris_1230002,Data Engineer,PARIS
3,KeyPers_Paris_1230003,Computer Engineer,PARIS
4,KeyPers_Paris_1230004,Computer Engineer,PARIS


#### Traitement journalier des données de matériel informatique

In [11]:
def extract_materiel(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    # Liste pour stocker les dataframes de chaque jour
    all_materiel_dfs = []

    # Boucle sur chaque jour
    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        # Liste pour stocker les dataframes de matériel de la journée
        materiel_day_dfs = []
        
        # Boucle sur chaque site
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_INFORMATIQUE/MATERIEL_INFORMATIQUE_{day_str}.txt"
            
            # Vérifier si le fichier existe avant de le lire
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                clean_date(df, site, "DATE_ACHAT")
                df['ID_DATE'] = df['DATE_ACHAT'].dt.date #pour jointure 
                materiel_day_dfs.append(df)
                
        # Si des fichiers ont été trouvés pour ce jour
        if materiel_day_dfs:
            # Combiner les données de tous les sites pour la journée et l'ajouter à la liste globale
            all_materiel_dfs.append(pd.concat(materiel_day_dfs, ignore_index=True))

    # Après la boucle, vérifier si on a collecté des données
    if all_materiel_dfs:
        # Combiner les données de tous les jours en un seul DataFrame
        materiel_total_df = pd.concat(all_materiel_dfs, ignore_index=True)
        return materiel_total_df
    else:
        print("Aucun fichier de matériel trouvé pour la période spécifiée.")

In [12]:
start_date = date(2026, 4, 29)
end_date = date(2026, 11, 14) 

mat_df = extract_materiel(start_date, end_date)
mat_df.head()

,ID_MATERIELINFO,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_ACHAT,TYPE,MODELE,ID_DATE
0,Paris_MATERIEL_INFO_202604290,KeyPers_Paris_1232362,Name2362,FistName2362,2026-04-29 08:17:31,PC fixe sans ecran,Z,2026-04-29
1,Paris_MATERIEL_INFO_202604291,KeyPers_Paris_1232165,Name2165,FistName2165,2026-04-29 09:42:55,imprimante,Laser A3 (>100kg),2026-04-29
2,Paris_MATERIEL_INFO_202604292,KeyPers_Paris_1231951,Name1951,FistName1951,2026-04-29 13:58:12,PC fixe sans ecran,Precision tower 3xxx,2026-04-29
3,Paris_MATERIEL_INFO_202604293,KeyPers_Paris_1230614,Name614,FistName614,2026-04-29 13:19:31,Telephone IP,,2026-04-29
4,Paris_MATERIEL_INFO_202604294,KeyPers_Paris_1232952,Name2952,FistName2952,2026-04-29 13:55:41,,modèle par défaut,2026-04-29


#### Traitement journalier des données de missions

In [13]:
# Même logique que la fonction d'extraction du matériel
def extract_missions(start_date, end_date):
    sites = ["PARIS", "BERLIN", "LONDON", "NEWYORK", "SHANGHAI", "LOSANGELES"]
    base_path = Path("data/BDD_BGES")
    
    all_missions_dfs = []

    for single_date in pd.date_range(start_date, end_date):
        day_str = single_date.strftime("%Y%m%d")
        
        missions_day_dfs = []
        
        for site in sites:
            file_path = base_path / f"BDD_BGES_{site}/BDD_BGES_{site}_MISSION/MISSION_{day_str}.txt"
            
            if file_path.exists():
                df = pd.read_csv(str(file_path), sep=';')
                clean_langue_mission(df, site)
                clean_date(df, site, "DATE_MISSION")
                df['ID_SITE'] = site
                df['ID_DATE'] = df['DATE_MISSION'].dt.date # pour jointure 
                missions_day_dfs.append(df)
                
        if missions_day_dfs:
            all_missions_dfs.append(pd.concat(missions_day_dfs, ignore_index=True))

    if all_missions_dfs :
        missions_total_df = pd.concat(all_missions_dfs, ignore_index=True)
        return missions_total_df
    else:
        print("Aucun fichier de missions trouvé pour la période spécifiée.")

In [14]:
missions_df = extract_missions(start_date, end_date)
missions_df.head()

C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])


C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])
C:\Users\etien\AppData\Local\Temp\ipykernel_16456\3652553356.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[date_col] = pd.to_datetime(df[date_col])


,ID_MISSION,ID_PERSONNEL,NOM_PERSONNEL,PRENOM_PERSONNEL,DATE_MISSION,TYPE_MISSION,VILLE_DEPART,PAYS_DEPART,VILLE_DESTINATION,PAYS_DESTINATION,TRANSPORT,ALLER_RETOUR,ID_SITE,ID_DATE
0,Paris_202604290,KeyPers_Paris_1233578,Name3578,FistName3578,2026-04-29 15:01:12,Business Meeting,Paris,France,New-York,USA,Avion,oui,PARIS,2026-04-29
1,Paris_202604291,KeyPers_Paris_1234968,Name4968,FistName4968,2026-04-29 18:41:44,Development,Paris,France,London,England,Avion,oui,PARIS,2026-04-29
2,Paris_202604292,KeyPers_Paris_1233110,Name3110,FistName3110,2026-04-29 09:13:25,Business Meeting,Paris,France,Washington,USA,Avion,oui,PARIS,2026-04-29
3,Paris_202604293,KeyPers_Paris_1230093,Name93,FistName93,2026-04-29 13:04:44,Conference,Paris,France,Berlin,Allemagne,Avion,oui,PARIS,2026-04-29
4,Paris_202604294,KeyPers_Paris_1234582,Name4582,FistName4582,2026-04-29 15:34:56,Development,Paris,France,Montreal,Canada,Avion,oui,PARIS,2026-04-29


Passage des données nettoyées au format Pyspark SQL Dataframe pour préparer leur chargement dans les tables finales (Load). 

In [15]:
sdf_materiel = spark.createDataFrame(mat_df)
sdf_personnel = spark.createDataFrame(personnel_df)
sdf_mission = spark.createDataFrame(missions_df)

#### Création tables de dimension

In [16]:
dim_materiel = (
    sdf_materiel
    .select("ID_MATERIELINFO", "TYPE", "MODELE")
    .withColumnRenamed("ID_MATERIELINFO", "ID_MATERIEL")
)

In [22]:
dim_materiel.show(5)

+--------------------+------------------+--------------------+
|         ID_MATERIEL|              TYPE|              MODELE|
+--------------------+------------------+--------------------+
|Paris_MATERIEL_IN...|PC fixe sans ecran|                   Z|
|Paris_MATERIEL_IN...|        imprimante|   Laser A3 (>100kg)|
|Paris_MATERIEL_IN...|PC fixe sans ecran|Precision tower 3xxx|
|Paris_MATERIEL_IN...|      Telephone IP|                    |
|Paris_MATERIEL_IN...|                  |   modèle par défaut|
+--------------------+------------------+--------------------+
only showing top 5 rows


In [17]:
data = [("BERLIN",), ("LONDON",), ("LOSANGELES",), ("NEWYORK",), ("PARIS",), ("SHANGHAI",)]
dim_site = spark.createDataFrame(data).toDF("ID_SITE")

In [24]:
dim_site.show(6)

+----------+
|   ID_SITE|
+----------+
|    BERLIN|
|    LONDON|
|LOSANGELES|
|   NEWYORK|
|     PARIS|
|  SHANGHAI|
+----------+



In [18]:
# Convertir toutes les dates au même format (date)
dates_missions = set(pd.to_datetime(missions_df['ID_DATE']))
dates_materiel = set(pd.to_datetime(mat_df['ID_DATE']))

# Combiner et trier
all_dates = sorted(dates_missions | dates_materiel)

# Créer la table de dimension
dim_date_data = [(d.date(),) for d in all_dates]
dim_date = spark.createDataFrame(dim_date_data, "ID_DATE date")

In [26]:
dim_date.show(5)

+----------+
|   ID_DATE|
+----------+
|2026-04-28|
|2026-04-29|
|2026-04-30|
|2026-05-01|
|2026-05-02|
+----------+
only showing top 5 rows


In [19]:
dim_personnel = sdf_personnel

In [22]:
dim_personnel.show(5)

+--------------------+------------------+-------+
|        ID_PERSONNEL|FONCTION_PERSONNEL|ID_SITE|
+--------------------+------------------+-------+
|KeyPers_Paris_123...|Business Executive|  PARIS|
|KeyPers_Paris_123...|               HRD|  PARIS|
|KeyPers_Paris_123...|     Data Engineer|  PARIS|
|KeyPers_Paris_123...| Computer Engineer|  PARIS|
|KeyPers_Paris_123...| Computer Engineer|  PARIS|
+--------------------+------------------+-------+
only showing top 5 rows


In [20]:
dim_mission = (
    sdf_mission
    .select(
        "ID_MISSION",
        "TYPE_MISSION",
        "VILLE_DEPART",
        "PAYS_DEPART",
        "VILLE_DESTINATION",
        "PAYS_DESTINATION",
        "TRANSPORT",
        "ALLER_RETOUR"
    )
)

In [23]:
dim_mission.show(5)

+---------------+----------------+------------+-----------+-----------------+----------------+---------+------------+
|     ID_MISSION|    TYPE_MISSION|VILLE_DEPART|PAYS_DEPART|VILLE_DESTINATION|PAYS_DESTINATION|TRANSPORT|ALLER_RETOUR|
+---------------+----------------+------------+-----------+-----------------+----------------+---------+------------+
|Paris_202604290|Business Meeting|       Paris|     France|         New-York|             USA|    Avion|         oui|
|Paris_202604291|     Development|       Paris|     France|           London|         England|    Avion|         oui|
|Paris_202604292|Business Meeting|       Paris|     France|       Washington|             USA|    Avion|         oui|
|Paris_202604293|      Conference|       Paris|     France|           Berlin|       Allemagne|    Avion|         oui|
|Paris_202604294|     Development|       Paris|     France|         Montreal|          Canada|    Avion|         oui|
+---------------+----------------+------------+---------

#### Création tables de faits

In [21]:
fait_materiel = (
    sdf_materiel
    .join(dim_date, "ID_DATE", "inner")
    .join(sdf_personnel, "ID_PERSONNEL", "inner")
    .join(dim_site, "ID_SITE", "inner")
    .select(col("ID_MATERIELINFO").alias("ID_MATERIEL"), "ID_PERSONNEL", "ID_SITE", col("ID_DATE").alias("ID_DATE_ACHAT"))
)

In [51]:
fait_materiel.show(5)

+--------------------+--------------------+-------+-------------+
|         ID_MATERIEL|        ID_PERSONNEL|ID_SITE|ID_DATE_ACHAT|
+--------------------+--------------------+-------+-------------+
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-23|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-22|
|BERLIN_MATERIEL_I...|KeyPers_Berlin_12...| BERLIN|   2026-05-22|
+--------------------+--------------------+-------+-------------+
only showing top 5 rows


In [22]:
fait_mission = (
    sdf_mission
    .join(dim_date, "ID_DATE", "inner")
    .join(sdf_personnel, "ID_PERSONNEL", "inner")
    .join(dim_site, "ID_SITE", "inner")
    .select("ID_MISSION", "ID_PERSONNEL", sdf_mission.ID_SITE, col("ID_DATE").alias("ID_DATE_MISSION"))
)

In [34]:
fait_mission.show(5)

+-----------------+--------------------+-------+---------------+
|       ID_MISSION|        ID_PERSONNEL|ID_SITE|ID_DATE_MISSION|
+-----------------+--------------------+-------+---------------+
|BERLIN_2026090627|KeyPers_Berlin_12...| BERLIN|     2026-09-06|
|BERLIN_2026052633|KeyPers_Berlin_12...| BERLIN|     2026-05-26|
| BERLIN_202605307|KeyPers_Berlin_12...| BERLIN|     2026-05-30|
| BERLIN_202605219|KeyPers_Berlin_12...| BERLIN|     2026-05-21|
|BERLIN_2026051616|KeyPers_Berlin_12...| BERLIN|     2026-05-16|
+-----------------+--------------------+-------+---------------+
only showing top 5 rows


## Réponse aux questions

1. Combien de cadres travaillent sur le site de Paris ?

In [23]:
nb_cadres_paris = (
    dim_personnel
    .filter(col("ID_SITE") == "PARIS")
    .filter(col("FONCTION_PERSONNEL") == "Business Executive")
    .count()
)

print(f"{nb_cadres_paris} cadres travaillent sur le site de Paris")

929 cadres travaillent sur le site de Paris


2. Combien d’ingénieurs Data travaillent sur les sites aux États-Unis ?

In [24]:
nb_inge_data_us = (
    dim_personnel
    .filter((col("ID_SITE") == "LOSANGELES") |  (col("ID_SITE") == "NEWYORK"))
    .filter(col("FONCTION_PERSONNEL") == "Data Engineer")
    .count()
)

print(f"{nb_inge_data_us} ingénieurs Data travaillent sur les sites aux États-Unis")

2197 ingénieurs Data travaillent sur les sites aux États-Unis


3. Combien d’ingénieurs informaticiens travaillent dans l’organisation (tous sites compris) ?

In [25]:
nb_inge_info = (
    dim_personnel
    .filter(col("FONCTION_PERSONNEL") == "Computer Engineer")
    .count()
)

print(f"{nb_inge_info} ingénieurs informaticiens travaillent dans l'organisation (tous sites compris)")

7696 ingénieurs informaticiens travaillent dans l'organisation (tous sites compris)


4. Combien de PC fixes ont été achetés par l’organisation entre juin et septembre 2026 ?

In [26]:
nb_pc_fixes = (
    dim_materiel
    .filter((col("TYPE") == "PC fixe sans ecran") | (col("TYPE") == "PC fixe tout-en-un"))
    .join(fait_materiel, "ID_MATERIEL")
    .filter((col("ID_DATE_ACHAT") >= date(2026, 6, 1)) & (col("ID_DATE_ACHAT") <= date(2026, 9, 30)))
    .count()
)

print(f"{nb_pc_fixes} PC fixes ont été achetés entre juin et septembre 2026")

1585 PC fixes ont été achetés entre juin et septembre 2026


5. Quelle a été l’impact carbone des PC fixes sans ecran entre mai et octobre 2026 ?

In [27]:
# On charge le fichier qui contient l'impact carbone du matériel informatique. 
import unicodedata

impact_mat_info_psdf = ps.read_csv("./data/BDD_BGES/materiel_informatique_impact.csv")
impact_mat_info_sdf = impact_mat_info_psdf.to_spark()

#Fonction pour retirer les accents et convertir en majuscules
def remove_accents_and_uppercase(name):
    #Normaliser et retirer les accents
    normalized = unicodedata.normalize('NFD', name)
    without_accents = ''.join(char for char in normalized if unicodedata.category(char) != 'Mn')
    return without_accents.upper()

# Convertir tous les noms de colonnes en majuscules sans accents pour pouvoir faire les jointures
impact_mat_info_sdf = impact_mat_info_sdf.select([col(c).alias(remove_accents_and_uppercase(c)) for c in impact_mat_info_sdf.columns])

impact_mat_info_sdf.show(5)

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+------------------+-----------------+------+
|              TYPE|           MODELE|IMPACT|
+------------------+-----------------+------+
|PC fixe sans ecran|modèle par défaut|   350|
|PC fixe sans ecran|   Optiplex micro|   174|
|PC fixe sans ecran|   Optiplex small|   240|
|PC fixe sans ecran|   Optiplex tower|   260|
|PC fixe sans ecran| Wyse thin client|    69|
+------------------+-----------------+------+
only showing top 5 rows


In [28]:
impact_pc_fixes = (
    dim_materiel
    .filter(col("TYPE") == "PC fixe sans ecran")
    .join(fait_materiel, "ID_MATERIEL")
    .filter((col("ID_DATE_ACHAT") >= date(2026, 5, 1)) & (col("ID_DATE_ACHAT") <= date(2026, 10, 31)))
    .join(impact_mat_info_sdf, ["TYPE", "MODELE"])
    .agg(sum(col("IMPACT")))
    .first()[0]
)

#On divise par 1000 car les unités dans le tableau du matériel sont en kgCO2e
print(f"L'impact des PC fixes sans ecran a été de {impact_pc_fixes/1000} tCO2e entre mai et octobre 2026")

L'impact des PC fixes sans ecran a été de 480.921 tCO2e entre mai et octobre 2026


6. Quelle a été l’impact carbone des PC portables achetés par les ingénieurs Data entre mai et octobre 2026 sur les sites de Londres et New-York ?

In [29]:
impact_pc_portables = (
    dim_personnel
    .filter((col("ID_SITE") == "LONDON") |  (col("ID_SITE") == "NEWYORK"))
    .filter(col("FONCTION_PERSONNEL") == "Data Engineer")
    .join(fait_materiel, "ID_PERSONNEL")
    .filter((col("ID_DATE_ACHAT") >= date(2026, 5, 1)) & (col("ID_DATE_ACHAT") <= date(2026, 10, 31)))
    .join(dim_materiel, "ID_MATERIEL")
    .filter(col("TYPE") == "PC portable")
    .join(impact_mat_info_sdf, ["TYPE", "MODELE"])
    .agg(sum(col("IMPACT")))
    .first()[0]
)

print(f"L'impact des PC portables achetés entre mai et octobre 2026 par les ingénieurs Data sur les sites de Londres et New-York a été de {impact_pc_portables/1000} tCO2e")

L'impact des PC portables achetés entre mai et octobre 2026 par les ingénieurs Data sur les sites de Londres et New-York a été de 61.723 tCO2e


7. Quelle a été l’impact carbone des Ecrans achetés par les cadres entre juillet et septembre 2026 sur tous les sites de l’organisation ?

In [30]:
impact_ecrans = (
    dim_personnel
    .filter(col("FONCTION_PERSONNEL") == "Business Executive")
    .join(fait_materiel, "ID_PERSONNEL")
    .filter((col("ID_DATE_ACHAT") >= date(2026, 7, 1)) & (col("ID_DATE_ACHAT") <= date(2026, 9, 30)))
    .join(dim_materiel, "ID_MATERIEL")
    .filter(col("TYPE") == "Ecran")
    .join(impact_mat_info_sdf, ["TYPE", "MODELE"])
    .agg(sum(col("IMPACT")))
    .first()[0]
)

print(f"L'impact des écrans achetés entre juillet et septembre 2026 par les cadres sur tous les sites a été de {impact_ecrans/1000} tCO2e")

L'impact des écrans achetés entre juillet et septembre 2026 par les cadres sur tous les sites a été de 11.8 tCO2e


8. Quelle a été l’impact carbone des missions sur les sites Européens entre mai et octobre 2026 ?

Clarification de "transports en communs" : 
Sur le site labos1.5, on a plusieurs types de transports en commun, dont deux qui ne sont pas pris en compte dans la liste donnée dans le sujet (bus et bateau). Cependant, quand on regarde dans la liste des missions, on constate que les missions labellisées transport en commun ont toutes la même ville d'arrivée et de départ, cf exécuter le code suivant : 

missions_df[(missions_df["TRANSPORT"] == "Transports en commun") & (missions_df["VILLE_DEPART"] != missions_df["VILLE_DESTINATION"])]

On considère donc que les trajets en transport peuvent être représentés par des trajets en bus. Sur le site de labo1.5 on trouve : 

Spécificité des déplacements en bus :
Nous faisons l’hypothèse que les trajets en bus inter-urbains seront majoritaires en terme de distances parcourues lors d’une mission, comparativement aux trajets en bus urbains. Les trajets inter-urbains étant moins émetteurs que les trajets urbains, nous prenons le facteur d’émission le moins élevé parmi les 3 facteurs disponibles dans la Base Carbone: c’est celui des bus dans des agglomérations de plus de 250 000 habitants.

Ce facteur sera donc utilisé pour les déplacements en transports en commun. 

Fonction permettant de calculer la distance entre 2 villes : 

In [ ]:
#Dictionnaire des coefficients en fonction du mode de transport
COEFFS_DISTANCE = {
    "Train" : 1.2,
    "Taxi" : 1.2,
    "Transports en commun" : 1.5
}

# Dictionnaire de coordonnées pour éviter les appels répétés et la limitation API GeoPy
CITY_COORDS = {
    # Sites principaux
    ("Paris", "France"): (48.8566, 2.3522),
    ("Berlin", "Germany"): (52.5200, 13.4050),
    ("Berlin", "Allemagne"): (52.5200, 13.4050),
    ("London", "England"): (51.5074, -0.1278),
    ("New York", "USA"): (40.7128, -74.0060),
    ("New-York", "USA"): (40.7128, -74.0060),
    ("Los Angeles", "USA"): (34.0522, -118.2437),
    ("Shanghai", "China"): (31.2304, 121.4737),
    
    # Autres villes européennes
    ("Marseille", "France"): (43.2965, 5.3698),
    ("Compiègne", "France"): (49.4144, 2.8259),
    ("Stockholm", "Sweden"): (59.3293, 18.0686),
    ("Stockholm", "Suède"): (59.3293, 18.0686),
    ("Helsinki", "Finland"): (60.1695, 24.9354),
    ("Helsinki", "Finlande"): (60.1695, 24.9354),
    
    # Villes asiatiques
    ("Osaka", "Japan"): (34.6937, 135.5023),
    ("Tokyo", "Japan"): (35.6762, 139.6503),
    ("Melbourne", "Australia"): (-37.8136, 144.9631),
    ("Sydney", "Australia"): (-33.8688, 151.2093),
    ("Sidney", "Australia"): (-33.8688, 151.2093),
    ("Wellington", "New Zealand"): (-41.2865, 174.7762),
    
    # Villes nord-américaines
    ("Montreal", "Canada"): (45.5017, -73.5673),
    ("Vancouver", "Canada"): (49.2827, -123.1207),
    ("Washington", "USA"): (38.9072, -77.0369),
    
    # Villes sud-américaines
    ("Buenos Aires", "Argentina"): (-34.6037, -58.3816),
    ("Bogota", "Colombia"): (4.7110, -74.0721),
    ("Rio de Janeiro", "Brazil"): (-22.9068, -43.1729),
    
    # Villes africaines/moyen-orient
    ("Rabat", "Morocco"): (34.0209, -6.8416),
    ("Rabat", "Maroc"): (34.0209, -6.8416),
    ("Dubaï", "Emirats"): (25.2048, 55.2708),
    ("Mexico", "Mexico"): (19.4326, -99.1332),
    ("Tunis", "Tunisia"): (36.8065, 10.1686),
    ("Tunis", "Tunisie"): (36.8065, 10.1686),
    ("Sao Paulo", "Brazil"): (-23.5505, -46.6333),
    ("Stockholm", "Sweden"): (59.3293, 18.0686),
    ("Stockholm", "Suede"): (59.3293, 18.0686),
    ("Alger", "Algeria"): (36.7538, 3.0588),
    ("Auckland", "New Zealand"): (-37.0882, 174.8860),
    ("Bordeaux", "France"): (44.8378, -0.5792),
    ("Pekin", "China"): (39.9042, 116.4074),
    ("Beijing", "China"): (39.9042, 116.4074),
    ("Lille", "France"): (50.6292, 3.0573),
    ("Oslo", "Norvège"): (59.9139, 10.7522),
    ("Oslo", "Norway"): (59.9139, 10.7522),
    ("Lima", "Peru"): (-12.0464, -77.0428)
}

def get_distance_between_cities(ville_depart, pays_depart, ville_destination, pays_destination):
    try:
        # Chercher dans le dictionnaire d'abord
        coords_depart = CITY_COORDS.get((ville_depart, pays_depart))
        coords_dest = CITY_COORDS.get((ville_destination, pays_destination))
        
        # Si ville de départ non trouvée, appeler Nominatim
        if not coords_depart:
            print("ville depart non présente: ", ville_depart)
            print("pays depart non présent: ", pays_depart)
            location_depart = geolocator.geocode(f"{ville_depart}, {pays_depart}", timeout=10)
            if location_depart:
                coords_depart = (location_depart.latitude, location_depart.longitude)
        
        # Si ville de destination non trouvée, appeler Nominatim
        if not coords_dest:
            print("ville arrivee non présente: ", ville_destination)
            print("pays arrivee non présent: ", pays_destination)
            location_destination = geolocator.geocode(f"{ville_destination}, {pays_destination}", timeout=10)
            if location_destination:
                coords_dest = (location_destination.latitude, location_destination.longitude)
        
        # Si les deux coordonnées ont été trouvées, calculer la distance
        if coords_depart and coords_dest:
            distance_km = geodesic(coords_depart, coords_dest).kilometers
            return distance_km
        else:
            return None
            
    except Exception as e:
        print(f"Erreur lors du calcul: {e}")
        return None

Fonction permettant de calculer les emissions : 

In [80]:
def get_emission(distance, transport, ar, pays_depart, pays_destination):
        
        #Chargement des fichiers permettant de récupérer les facteurs d'émission
        base_path = Path("data/facteurs_emission")
        fe_transports_en_commun_df = pd.read_csv(base_path / "fe_transports_en_commun.tsv", sep='\t')
        fe_vehicules_df = pd.read_csv(base_path / "fe_vehicules.tsv", sep='\t')
        fe_transports_en_commun_df.head(100)


        #Multiplicateur si le trajet est un aller-retour
        multAr = 2 if ar == "oui" else 1

        #Modification de la distance en fonction du moyen de transport
        if(transport == "Avion"):
            distance += 95
        else : 
            distance *= COEFFS_DISTANCE.get(transport)
        
        #Calcul du facteur et de l'emission totale en fonction du moyen de transport
        #On divise par 1000 à chaque fois pour renvoyer la quantité directement en tCO2e
        match transport:
            case "Avion":
                if distance < 1000:
                    facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Court courrier (< 1000 km)"]["total"].iloc[0]
                elif distance < 3500:
                    facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Moyen courrier (< 1001 - 3500km)"]["total"].iloc[0]
                else:
                    facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Long courrier (> 3500 km)"]["total"].iloc[0]
                
                return distance*facteur*multAr/1000

            
            case "Train":
                if(pays_depart == "France"):
                    if(pays_destination == "France"):
                        if distance > 200 :
                            facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "TGV > 200 km"]["total"].iloc[0]
                        else:
                            facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Train < 200 km"]["total"].iloc[0]
                    else:
                        facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Train mixte France et international"]["total"].iloc[0]
                else:
                    if(pays_destination == "France"):
                        facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Train mixte France et international"]["total"].iloc[0]
                    else:
                        facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Train international"]["total"].iloc[0]
                
                return distance*facteur*multAr/1000
            
            case "Taxi":
                facteur = fe_vehicules_df[fe_vehicules_df["subsubcategory"] == "Motorisation inconnue"]["total"].iloc[0]
                #On calcule aussi les émissions pour le trajet aller (on condisère 2 personnes à bord)
                #et retour (on considère le chauffeur seul à bord)
                em = (distance * (1 + 1/2) * facteur) + (distance * (1 + 1/1) * facteur)
                return multAr*em/1000
            
            case "Transports en commun":
                facteur = fe_transports_en_commun_df[fe_transports_en_commun_df["subsubcategory"] == "Bus > 250 000 habitants"]["total"].iloc[0]
                return distance*facteur*multAr/1000

Réponse à la question :

In [ ]:
#On isole les missions concernées
missions = (
    dim_mission
    .join(fait_mission, "ID_MISSION")
    .filter((col("ID_SITE") == "PARIS") | (col("ID_SITE") == "LONDON") | (col("ID_SITE") == "BERLIN"))
    .filter((col("ID_DATE_MISSION") >= date(2026, 5, 1)) & (col("ID_DATE_MISSION") <= date(2026, 10, 31)))
    .select("VILLE_DEPART","PAYS_DEPART", "VILLE_DESTINATION","PAYS_DESTINATION","ALLER_RETOUR","TRANSPORT")
)

#On converti en Dataframe Pandas pour les calculs
missions_df = missions.toPandas()

In [50]:
#On applique la fonction de calcul des distances à chaque mission et on sauvegarde le résultat
#dans une nouvelle colonne.
missions_df['DISTANCE_KM'] = missions_df.apply(
    lambda row: get_distance_between_cities(
        row['VILLE_DEPART'],
        row['PAYS_DEPART'],
        row['VILLE_DESTINATION'],
        row['PAYS_DESTINATION'],
    ),
    axis=1
)

In [ ]:
#On applique la fonction de calcul des emissions et on sauvegarde le résultat dans une nouvelle colonne
missions_df['EMISSION'] = missions_df.apply(
    lambda row: get_emission(
        row['DISTANCE_KM'],
        row['TRANSPORT'],
        row['ALLER_RETOUR'],
        row['PAYS_DEPART'],
        row['PAYS_DESTINATION']
    ),
    axis=1
)

In [81]:
total_emission = missions_df['EMISSION'].sum()
print(f"L’impact carbone des missions sur les sites Européens entre mai et octobre 2026 est de {total_emission:.2f} tCO2e.")

L’impact carbone des missions sur les sites Européens entre mai et octobre 2026 est de 25732.77 tCO2e.


9. Quels ont été les 5 jours les plus impactents concernant les missions en avion pour les sites Européens de l’organisation ?

In [87]:
missions_avion = (
    dim_mission
    .filter(col("TRANSPORT") == "Avion")
    .join(fait_mission, "ID_MISSION")
    .filter((col("ID_SITE") == "PARIS") | (col("ID_SITE") == "LONDON") | (col("ID_SITE") == "BERLIN"))
    .select("VILLE_DEPART","PAYS_DEPART", "VILLE_DESTINATION","PAYS_DESTINATION","ALLER_RETOUR","TRANSPORT","ID_DATE_MISSION")
)

missions_df = missions_avion.toPandas()

In [88]:
#Calcul des émissions pour chaque mission
missions_df['DISTANCE_KM'] = missions_df.apply(
    lambda row: get_distance_between_cities(
        row['VILLE_DEPART'],
        row['PAYS_DEPART'],
        row['VILLE_DESTINATION'],
        row['PAYS_DESTINATION'],
    ),
    axis=1
)

missions_df['EMISSION'] = missions_df.apply(
    lambda row: get_emission(
        row['DISTANCE_KM'],
        row['TRANSPORT'],
        row['ALLER_RETOUR'],
        row['PAYS_DEPART'],
        row['PAYS_DESTINATION']
    ),
    axis=1
)

In [92]:
# Grouper par date et sommer les émissions
emissions_par_jour = missions_df.groupby('ID_DATE_MISSION')['EMISSION'].sum().reset_index()
emissions_par_jour.columns = ['Date', 'Emission_tCO2e']

# Trier par émissions décroissantes et prendre les 5 premiers
top_5_jours = emissions_par_jour.nlargest(5, 'Emission_tCO2e').reset_index(drop=True)

# Afficher le résultats
print("Les 5 jours les plus impactants pour les missions en avion sur les sites européens sont :\n")
print(top_5_jours.to_string(index=False))

Les 5 jours les plus impactants pour les missions en avion sur les sites européens sont :

      Date  Emission_tCO2e
2026-08-05      235.700371
2026-05-15      217.388351
2026-05-30      217.320879
2026-07-22      215.176076
2026-10-04      210.852861
